# Lesson 5 — Autograd & Computational Graph

## 学习目标

这一节开始进入神经网络训练真正的核心机制：自动微分。

前面我们一直在研究：

- Tensor 是什么；
- Shape 如何变化；
- Matrix Multiplication 如何工作；
- Attention 中 Tensor 如何 contraction。

但训练神经网络还需要解决另一个问题：

> 参数应该往哪个方向更新？

这需要梯度。

完成本节后，应能够：

1. 理解导数和梯度在神经网络训练中的作用；
2. 理解 `requires_grad=True`；
3. 理解 Computational Graph；
4. 理解 `backward()` 在做什么；
5. 理解 `.grad` 保存的是什么；
6. 理解 Chain Rule 与反向传播的关系；
7. 理解 Leaf Tensor；
8. 理解 PyTorch 为什么默认累积梯度；
9. 理解为什么训练时需要清空梯度；
10. 理解 `torch.no_grad()` 和 `detach()`；
11. 理解为什么通常对 scalar loss 调用 `backward()`；
12. 从头看懂：

`loss.backward()`

到底发生了什么。


## 1. 为什么需要梯度？

假设模型中只有一个参数：

$$
w
$$

Loss Function：

$$
L(w)
$$

训练的目标是找到一个参数：

$$
w^*
$$

使：

$$
L(w^*)
$$

尽可能小。

如果知道：

$$
\frac{dL}{dw}
$$

就能够判断：

> 当 w 增大时，Loss 是增加还是减少。

如果：

$$
\frac{dL}{dw}>0
$$

说明增加 $w$ 会使 Loss 增大。

因此应该减小 $w$。

如果：

$$
\frac{dL}{dw}<0
$$

说明增加 $w$ 会使 Loss 减小。

因此应该增大 $w$。

Gradient Descent 的基本更新：

$$
w
\leftarrow
w-\eta\frac{dL}{dw}
$$

其中：

$$
\eta
$$

是 Learning Rate。

因此训练神经网络的核心循环，本质上就是：

Forward

↓

计算 Loss

↓

计算 Gradient

↓

更新 Parameters


## 2. 一个最简单的例子

考虑：

$$
y=x^2
$$

根据微积分：

$$
\frac{dy}{dx}=2x
$$

如果：

$$
x=3
$$

那么：

$$
\frac{dy}{dx}
=
2\times3
=
6
$$

PyTorch Autograd 的目标就是：

> 根据我们执行过的 Tensor 运算，自动得到这个导数。


In [1]:
import torch

x = torch.tensor(3.0, requires_grad=True)

y = x**2

y.backward()

print("x =", x)
print("y =", y)
print("dy/dx =", x.grad)


x = tensor(3., requires_grad=True)
y = tensor(9., grad_fn=<PowBackward0>)
dy/dx = tensor(6.)


## 3. `requires_grad=True`

创建 Tensor 时：

`requires_grad=True`

表示：

> PyTorch 需要记录与这个 Tensor 有关的运算，以便后续计算梯度。

例如：

`x = torch.tensor(3.0, requires_grad=True)`

之后：

`y = x ** 2`

PyTorch 不仅计算：

$$
y=9
$$

还会记录：

$$
x
\rightarrow
x^2
\rightarrow
y
$$

这个运算关系。

因此之后调用：

`y.backward()`

PyTorch 才知道如何计算：

$$
\frac{dy}{dx}
$$

如果 `requires_grad=False`，PyTorch 通常不会为这个 Tensor 构建用于梯度计算的完整记录。


In [ ]:
x1 = torch.tensor(3.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=False)

y1 = x1**2
y2 = x2**2

print("x1.requires_grad:", x1.requires_grad)
print("y1.requires_grad:", y1.requires_grad)
print("x2.requires_grad:", x2.requires_grad)
print("y2.requires_grad:", y2.requires_grad)


x1.requires_grad: True
y1.requires_grad: True
x2.requires_grad: False
y2.requires_grad: False


## 4. Computational Graph

考虑：

$$
x=2
$$

然后：

$$
a=x^2
$$

$$
b=3a
$$

$$
y=b+1
$$

Forward Computation：

$$
x
\rightarrow
a=x^2
\rightarrow
b=3a
\rightarrow
y=b+1
$$

这可以表示成一个 Computational Graph：

$$
x
\rightarrow
x^2
\rightarrow
a
\rightarrow
\times3
\rightarrow
b
\rightarrow
+1
\rightarrow
y
$$

PyTorch 在执行这些运算时，会动态记录：

- 哪些 Tensor 参与了运算；
- 执行了什么操作；
- 输出依赖哪些输入。

然后反向计算：

$$
\frac{dy}{dx}
$$

这就是 Autograd 的基础。


In [3]:
x = torch.tensor(2.0, requires_grad=True)

a = x**2
b = 3 * a
y = b + 1

print("x =", x)
print("a =", a)
print("b =", b)
print("y =", y)

print()
print("x.grad_fn:", x.grad_fn)
print("a.grad_fn:", a.grad_fn)
print("b.grad_fn:", b.grad_fn)
print("y.grad_fn:", y.grad_fn)


x = tensor(2., requires_grad=True)
a = tensor(4., grad_fn=<PowBackward0>)
b = tensor(12., grad_fn=<MulBackward0>)
y = tensor(13., grad_fn=<AddBackward0>)

x.grad_fn: None
a.grad_fn: <PowBackward0 object at 0x7eeb546c4760>
b.grad_fn: <MulBackward0 object at 0x7eeb546c4760>
y.grad_fn: <AddBackward0 object at 0x7eeb546c4760>


## 5. Chain Rule

Autograd 最核心的数学基础就是 Chain Rule。

刚才：

$$
a=x^2
$$

$$
b=3a
$$

$$
y=b+1
$$

我们要求：

$$
\frac{dy}{dx}
$$

根据 Chain Rule：

$$
\frac{dy}{dx}
=
\frac{dy}{db}
\frac{db}{da}
\frac{da}{dx}
$$

分别计算：

$$
\frac{dy}{db}=1
$$

$$
\frac{db}{da}=3
$$

$$
\frac{da}{dx}=2x
$$

所以：

$$
\frac{dy}{dx}
=
1\times3\times2x
$$

即：

$$
\frac{dy}{dx}=6x
$$

当：

$$
x=2
$$

得到：

$$
\frac{dy}{dx}=12
$$

Autograd 本质上就是自动执行这一整套 Chain Rule。


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

a = x**2
b = 3 * a
y = b + 1

y.backward()

print("PyTorch gradient:", x.grad)
print("Manual gradient:", 6 * x.item())


PyTorch gradient: tensor(12.)
Manual gradient: 12.0


## 6. Forward 与 Backward

Forward Pass：

$$
x
\rightarrow
a
\rightarrow
b
\rightarrow
y
$$

我们按照正常程序执行方向计算数值。

Backward Pass 则反过来：

$$
y
\rightarrow
b
\rightarrow
a
\rightarrow
x
$$

计算：

$$
\frac{\partial y}{\partial y}
$$

然后：

$$
\frac{\partial y}{\partial b}
$$

再：

$$
\frac{\partial y}{\partial a}
$$

最后：

$$
\frac{\partial y}{\partial x}
$$

所以所谓 Backpropagation，可以理解为：

> 从最终输出或 Loss 开始，沿 Computational Graph 反方向传播梯度。


## 7. 多变量函数

真实神经网络当然不会只有一个参数。

考虑：

$$
z=x^2+3y
$$

那么有两个偏导数：

$$
\frac{\partial z}{\partial x}=2x
$$

以及：

$$
\frac{\partial z}{\partial y}=3
$$

如果：

$$
x=2,\quad y=4
$$

那么：

$$
\frac{\partial z}{\partial x}=4
$$

$$
\frac{\partial z}{\partial y}=3
$$

Gradient 可以写为：

$$
\nabla z
=
\begin{bmatrix}
\frac{\partial z}{\partial x}\\
\frac{\partial z}{\partial y}
\end{bmatrix}
$$

也就是：

$$
\nabla z
=
\begin{bmatrix}
4\\
3
\end{bmatrix}
$$


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)
z = x**2 + 3 * y

z.backward()

print("z =", z.item())
print("dz/dx =", x.grad)
print("dz/dy =", y.grad)


z = 16.0
dz/dx = tensor(4.)
dz/dy = tensor(3.)


## 8. Gradient 本身也是 Tensor

神经网络中的参数通常不是 scalar。

例如：

$$
W.shape=(D,H)
$$

如果 Loss 是：

$$
L
$$

那么：

$$
\frac{\partial L}{\partial W}
$$

必须告诉我们：

> W 中每一个元素应该如何变化。

因此：

$$
W.grad.shape=W.shape
$$

例如：

$$
W.shape=(4,6)
$$

那么：

$$
W.grad.shape=(4,6)
$$

这条规则非常重要：

> Parameter 的 gradient 通常与 Parameter 本身具有相同 shape。


In [ ]:
D = 4
H = 6

W = torch.randn(D, H, requires_grad=True)
x = torch.randn(D)
y = x @ W

loss = y.sum()
loss.backward()

print("W.shape    :", W.shape)
print("W.grad.shape:", W.grad.shape)


W.shape    : torch.Size([4, 6])
W.grad.shape: torch.Size([4, 6])


## 9. Linear Layer 中的梯度

前面我们已经学过：

$$
Y=XW+b
$$

其中：

$$
X.shape=(B,T,D)
$$

$$
W.shape=(D,H)
$$

$$
b.shape=(H,)
$$

输出：

$$
Y.shape=(B,T,H)
$$

假设最后得到一个 scalar Loss：

$$
L
$$

Backward 之后：

$$
W.grad.shape=(D,H)
$$

$$
b.grad.shape=(H,)
$$

这意味着：

> Forward Pass 决定输出是什么；

而：

> Backward Pass 计算每个参数对最终 Loss 的影响。


In [ ]:
B = 2
T = 3
D = 4
H = 6

x = torch.randn(B, T, D)

weight = torch.randn(D, H, requires_grad=True)
bias = torch.randn(H, requires_grad=True)

y = x @ weight + bias

loss = y.pow(2).mean()
loss.backward()

print("output:", y.shape)
print("loss:", loss.shape)
print("weight:", weight.shape)
print("weight.grad:", weight.grad.shape)
print("bias:", bias.shape)
print("bias.grad:", bias.grad.shape)


output: torch.Size([2, 3, 6])
loss: torch.Size([])
weight: torch.Size([4, 6])
weight.grad: torch.Size([4, 6])
bias: torch.Size([6])
bias.grad: torch.Size([6])


## 10. 为什么通常对 Scalar Loss 调用 backward？

假设：

$$
loss
$$

是一个 scalar：

$$
loss.shape=()
$$

调用：

`loss.backward()`

意思非常明确：

> 计算这个 scalar loss 对所有相关参数的梯度。

例如：

$$
\frac{\partial L}{\partial W}
$$

但是如果：

$$
y.shape=(3,)
$$

那么：

`y.backward()`

就产生一个问题：

> 你到底希望对 y 的三个元素怎样组合之后求梯度？

因此直接对非 scalar Tensor 调用 `backward()`，通常需要额外提供上游梯度。

在标准神经网络训练中，我们通常先把多个样本、token 等对应的 loss reduction 成一个 scalar：

$$
L
$$

然后：

`L.backward()`

所以训练代码里最常见的是：

`loss.backward()`

而不是：

`logits.backward()`


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.stack([x, x**2, x**3])

print("y.shape:", y.shape)

try:
    y.backward()
except RuntimeError as error:
    print(error)


y.shape: torch.Size([3])
grad can be implicitly created only for scalar outputs


## 11. Vector-Jacobian Product

对于非 scalar 输出，可以显式提供：

`gradient=...`

例如：

$$
y=
\begin{bmatrix}
y_1\\
y_2\\
y_3
\end{bmatrix}
$$

调用：

`y.backward(v)`

实际上计算的是一种 Vector-Jacobian Product：

$$
v^\top
\frac{\partial y}{\partial x}
$$

现在不需要深入 Jacobian 的完整理论。

只需要知道：

> `backward()` 最自然的情况是 scalar output。

这也是为什么训练神经网络时通常先构造 scalar loss。

后面的 CS336 中，我们绝大多数时候都会：

`loss.backward()`


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.stack([x, x**2, x**3])

upstream = torch.ones_like(y)

y.backward(gradient=upstream)

print("x.grad =", x.grad)


x.grad = tensor(17.)


## 12. PyTorch 默认累积 Gradient

这是非常重要的一点。

PyTorch 默认不会在每次：

`backward()`

之前自动清除 `.grad`。

Gradient 会累积。

例如第一次 backward：

$$
x.grad=4
$$

再次 backward：

$$
x.grad=8
$$

而不是仍然等于：

$$
4
$$

原因是 PyTorch 支持很多需要累积梯度的场景。

所以训练神经网络时必须主动清理上一轮梯度。


In [10]:
x = torch.tensor(2.0, requires_grad=True)

y = x**2

y.backward()

print("After first backward:", x.grad)

y = x**2

y.backward()

print("After second backward:", x.grad)


After first backward: tensor(4.)
After second backward: tensor(8.)


## 13. 为什么训练循环需要清梯度？

假设某一步训练得到：

$$
\nabla L_1
$$

下一步得到：

$$
\nabla L_2
$$

如果没有清空梯度，PyTorch 会得到：

$$
\nabla L_1+\nabla L_2
$$

但标准训练通常希望第二步只使用：

$$
\nabla L_2
$$

因此每次训练 iteration 通常都会先清除旧梯度。

以后使用 Optimizer 时，会看到：

`optimizer.zero_grad()`

现在我们还没有正式学习 Optimizer。

手动 Tensor 可以：

`x.grad = None`

把梯度清掉。


In [11]:
x = torch.tensor(2.0, requires_grad=True)

# Step 1
loss = x**2
loss.backward()

print("Step 1 grad:", x.grad)

# Clear gradient
x.grad = None

# Step 2
loss = x**2
loss.backward()

print("Step 2 grad:", x.grad)


Step 1 grad: tensor(4.)
Step 2 grad: tensor(4.)


## 14. Gradient 为 None 和 Gradient 为 0

这两个状态不是完全同一个概念。

### `grad is None`

表示：

> 当前没有存储 gradient。

例如：

`x.grad = None`

### `grad == 0`

表示：

> Gradient Tensor 已经存在，只是其中数值全部为 0。

例如：

`x.grad.zero_()`

两种方式都可以用于清除梯度值。

后面使用 PyTorch Optimizer 时，我们通常使用：

`optimizer.zero_grad(set_to_none=True)`

或者：

`optimizer.zero_grad()`

具体差异后面学习 Optimizer 时再深入。

当前只需要形成：

> backward 默认累积 gradient，所以每一步训练前必须考虑 gradient reset。


## 15. Leaf Tensor

考虑：

`x = torch.tensor(2.0, requires_grad=True)`

然后：

`y = x ** 2`

这里：

`x`

通常是 Leaf Tensor。

因为它不是由另一个需要梯度的 Tensor 运算得到的。

而：

`y`

是通过：

$$
y=x^2
$$

计算得到。

所以 `y` 是 intermediate Tensor。

默认情况下，PyTorch 会重点把梯度保存在需要梯度的 Leaf Tensor 的 `.grad` 中。

因此：

`x.grad`

通常存在。

而 intermediate Tensor 的 `.grad` 默认不一定被保留。


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x**2
z = 3 * y

print("x.is_leaf:", x.is_leaf)
print("y.is_leaf:", y.is_leaf)
print("z.is_leaf:", z.is_leaf)

z.backward()

print("x.grad:", x.grad)
print("y.grad:", y.grad)


x.is_leaf: True
y.is_leaf: False
z.is_leaf: False
x.grad: tensor(12.)
y.grad: None


/tmp/ipykernel_7880/4004289071.py:16: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print("y.grad:", y.grad)


## 16. `retain_grad()`

有时候为了 debug 或学习，我们确实希望查看 intermediate Tensor 的 gradient。

可以：

`y.retain_grad()`

告诉 PyTorch：

> backward 后也把 y 的 gradient 保存下来。

例如：

$$
y=x^2
$$

$$
z=3y
$$

那么：

$$
\frac{dz}{dy}=3
$$

使用 `retain_grad()` 后，可以直接查看：

`y.grad`

得到这个值。

正式训练中通常不需要给大量 intermediate Tensor 保留 gradient，因为这会增加内存占用。

但在学习 Autograd 时非常有帮助。


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2

y.retain_grad()

z = 3 * y

z.backward()

print("dz/dx:", x.grad)
print("dz/dy:", y.grad)


dz/dx: tensor(12.)
dz/dy: tensor(3.)


## 17. Dynamic Computational Graph

PyTorch 的一个重要特点是：

> Graph 是随着 Python 代码实际执行动态建立的。

例如：

`if`

`for`

不同输入走不同路径，都可以构建不同的计算图。

这意味着：

Forward Pass 不只是计算数值。

同时也在建立后续 backward 所需要的运算关系。

因此可以粗略理解：

Forward：

$$
\text{compute values}
+
\text{record graph}
$$

Backward：

$$
\text{traverse graph backward}
+
\text{apply chain rule}
$$


In [ ]:
def compute(x: torch.Tensor) -> torch.Tensor:
    if x.item() > 0:
        return x**2

    return x**3


x = torch.tensor(2.0, requires_grad=True)
y = compute(x)

y.backward()

print("y =", y)
print("grad =", x.grad)


y = tensor(4., grad_fn=<PowBackward0>)
grad = tensor(4.)


## 18. Graph 生命周期

考虑：

`y = x ** 2`

然后：

`y.backward()`

PyTorch 在 backward 时，为了节省内存，通常会释放部分用于反向传播的中间信息。

因此对同一个 graph 再次：

`y.backward()`

通常会报错。

如果确实需要在同一个 graph 上 backward 多次，可以使用：

`retain_graph=True`

但标准神经网络训练通常不会这样做。

正常训练过程是：

Forward

↓

创建新的 graph

↓

Backward

↓

释放 graph

↓

下一轮重新 Forward

所以不要养成随意添加：

`retain_graph=True`

来“修复报错”的习惯。

如果需要它，应该明确知道为什么需要。


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2

y.backward()

try:
    y.backward()
except RuntimeError as error:
    print(error)


Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.


## 19. `torch.no_grad()`

训练时需要 Autograd。

但很多情况下我们只想做 Forward，而不想构建梯度图。

例如：

- Validation
- Inference
- 参数手动更新
- 某些不需要 gradient 的计算

这时可以：

`with torch.no_grad():`

在这个 context 中执行的运算不会正常加入梯度追踪。

例如：

`y = model(x)`

如果只是在 inference，就没有必要保存完整 backward graph。

这样通常可以减少内存开销。


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y1 = x**2

with torch.no_grad():
    y2 = x**2

print("y1.requires_grad:", y1.requires_grad)
print("y2.requires_grad:", y2.requires_grad)


y1.requires_grad: True
y2.requires_grad: False


## 20. 参数更新本身通常不应该进入 Graph

Gradient Descent：

$$
w
\leftarrow
w-\eta\nabla_wL
$$

更新参数这一步通常不需要 Autograd 再去追踪。

否则会变成：

> 用 Autograd 记录“参数是如何由上一次参数更新得到的”。

标准训练通常并不需要这一层 graph。

所以手写 Gradient Descent 时，可以：

`with torch.no_grad():`

然后更新：

`w -= lr * w.grad`

后面正式学习 Optimizer 后，这个细节会由 PyTorch Optimizer 帮我们处理。


In [ ]:
w = torch.tensor(2.0, requires_grad=True)

learning_rate = 0.1

loss = w**2
loss.backward()

print("Before update:", w.item())
print("Gradient:", w.grad.item())

with torch.no_grad():
    w -= learning_rate * w.grad

print("After update:", w.item())


Before update: 2.0
Gradient: 4.0
After update: 1.600000023841858


## 21. `detach()`

`detach()` 的作用可以粗略理解为：

> 得到一个与原 Tensor 共享数据关系、但脱离当前 Autograd Graph 的 Tensor view。

例如：

`y = x ** 2`

这里：

`y.requires_grad == True`

执行：

`z = y.detach()`

则：

`z.requires_grad == False`

后续使用 `z` 的运算不会沿原来的路径把 gradient 继续传回 x。

这在：

- Logging
- 某些 target computation
- 停止某条 gradient path

等场景中会使用。

当前最重要的是理解：

> `detach()` 会切断这一条 Autograd 路径。


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2

z = y.detach()

print("y.requires_grad:", y.requires_grad)
print("z.requires_grad:", z.requires_grad)


y.requires_grad: True
z.requires_grad: False


## 22. `detach()` vs `torch.no_grad()`

它们都可以让某些计算不继续被 Autograd 正常追踪，但使用方式不同。

### `detach()`

作用于一个 Tensor：

`z = y.detach()`

可以理解为：

> 从当前 Tensor 开始切断 gradient path。

---

### `torch.no_grad()`

作用于一个代码区域：

`with torch.no_grad():`

其中执行的运算不构建正常的梯度追踪。

可以粗略记：

`detach()`

→ Tensor-level

`no_grad()`

→ Context-level

目前理解到这个程度即可。


## 23. Gradient Flow

神经网络通常由很多层组成：

$$
x
\rightarrow
h_1
\rightarrow
h_2
\rightarrow
h_3
\rightarrow
L
$$

Backward：

$$
L
\rightarrow
h_3
\rightarrow
h_2
\rightarrow
h_1
\rightarrow
parameters
$$

每一层根据 Chain Rule 接收：

> upstream gradient

然后计算：

1. 对自己输入的 gradient；
2. 对自己 parameters 的 gradient。

最终：

$$
L.backward()
$$

会沿整张 Computational Graph 一直传播到所有相关的 trainable parameters。

所以训练一个 Transformer 时虽然只有：

`loss.backward()`

一行，

背后实际上会经过：

- LM Head
- Transformer Blocks
- SwiGLU
- RMSNorm
- Attention
- Q/K/V Projection
- Embedding

等大量操作。

Autograd 会自动完成整个 Chain Rule。


## 24. 最小训练循环的数学结构

现在已经可以理解一个非常简化的训练循环。

### Step 1：Forward

$$
\hat{y}=f(x;w)
$$

### Step 2：Loss

$$
L=L(\hat{y},y)
$$

### Step 3：Backward

`loss.backward()`

得到：

$$
\frac{\partial L}{\partial w}
$$

### Step 4：Parameter Update

$$
w
\leftarrow
w-\eta
\frac{\partial L}{\partial w}
$$

### Step 5：Clear Gradient

清掉旧 gradient。

然后重新开始下一轮。

完整循环：

Forward

↓

Loss

↓

Backward

↓

Gradient

↓

Update

↓

Clear Gradient

↓

Next Forward


## 25. 一个最小训练问题

假设真实关系：

$$
y=3x
$$

但我们的模型只有一个参数：

$$
\hat y=wx
$$

目标是通过训练让：

$$
w
$$

逐渐接近：

$$
3
$$

Loss 使用：

$$
L=(\hat y-y)^2
$$

也就是 Mean Squared Error 的简化版本。

这个例子会第一次完整连接：

- Forward
- Loss
- Backward
- Gradient
- Parameter Update


In [ ]:
x = torch.tensor(2.0)

target = torch.tensor(6.0)

w = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.1

for step in range(10):
    prediction = w * x

    loss = (prediction - target) ** 2
    w.grad = None
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad

    print(f"step={step:2d}", f"w={w.item():.4f}", f"loss={loss.item():.4f}")


step= 0 w=2.4000 loss=36.0000
step= 1 w=2.8800 loss=1.4400
step= 2 w=2.9760 loss=0.0576
step= 3 w=2.9952 loss=0.0023
step= 4 w=2.9990 loss=0.0001
step= 5 w=2.9998 loss=0.0000
step= 6 w=3.0000 loss=0.0000
step= 7 w=3.0000 loss=0.0000
step= 8 w=3.0000 loss=0.0000
step= 9 w=3.0000 loss=0.0000


## 26. Training Loop 中清梯度的位置

前面的最小循环写成：

`w.grad = None`

↓

Forward

↓

Loss

↓

Backward

也可以调整代码结构。

更常见的工程顺序是：

Clear Gradient

↓

Forward

↓

Loss

↓

Backward

↓

Update

也就是未来会看到：

`optimizer.zero_grad()`

`logits = model(x)`

`loss = ...`

`loss.backward()`

`optimizer.step()`

为什么先清梯度？

因为：

> backward 会把新的 gradient 加到已有 `.grad` 上。

所以必须确保当前 iteration 开始前没有上一轮残留 gradient。


In [ ]:
# 更标准的 Training Loop
x = torch.tensor(2.0)

target = torch.tensor(6.0)

w = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.1

for step in range(10):
    # 1. Clear old gradient
    w.grad = None

    # 2. Forward
    prediction = w * x

    # 3. Loss
    loss = (prediction - target) ** 2

    # 4. Backward
    loss.backward()

    # 5. Update
    with torch.no_grad():
        w -= learning_rate * w.grad

    print(
        f"step={step:2d}",
        f"w={w.item():.4f}",
        f"grad={w.grad.item():.4f}",
        f"loss={loss.item():.4f}",
    )


step= 0 w=2.4000 grad=-24.0000 loss=36.0000
step= 1 w=2.8800 grad=-4.8000 loss=1.4400
step= 2 w=2.9760 grad=-0.9600 loss=0.0576
step= 3 w=2.9952 grad=-0.1920 loss=0.0023
step= 4 w=2.9990 grad=-0.0384 loss=0.0001
step= 5 w=2.9998 grad=-0.0077 loss=0.0000
step= 6 w=3.0000 grad=-0.0015 loss=0.0000
step= 7 w=3.0000 grad=-0.0003 loss=0.0000
step= 8 w=3.0000 grad=-0.0001 loss=0.0000
step= 9 w=3.0000 grad=-0.0000 loss=0.0000


## 28. 不要丢掉 Shape Thinking

学习 Autograd 后，不要忘记前面建立的 Shape Thinking。

假设：

$$
X.shape=(B,T,D)
$$

$$
W_Q.shape=(D,D)
$$

Forward：

$$
Q=XW_Q
$$

所以：

$$
Q.shape=(B,T,D)
$$

最终得到 scalar：

$$
L
$$

Backward：

$$
\frac{\partial L}{\partial W_Q}
$$

必须和：

$$
W_Q
$$

具有相同 shape。

因此：

$$
W_Q.grad.shape=(D,D)
$$

同理，如果：

$$
W_{up}.shape=(D,4D)
$$

那么：

$$
W_{up}.grad.shape=(D,4D)
$$

所以从现在开始要同时追踪两个方向：

### Forward Shape

Tensor 的数值如何流动。

### Backward Shape

每个 Parameter 对应的 gradient 是什么 shape。


## 29. Attention 与 Autograd

上一节已经得到：

$$
scores
=
\frac{QK^\top}{\sqrt{D_h}}
$$

然后：

$$
A=\operatorname{softmax}(scores)
$$

最后：

$$
O=AV
$$

假设最终 Loss：

$$
L
$$

依赖 $O$。

Backward 时：

$$
L
\rightarrow
O
\rightarrow
A,V
$$

然后：

$$
A
\rightarrow
scores
$$

再：

$$
scores
\rightarrow
Q,K
$$

所以最终会得到：

$$
\frac{\partial L}{\partial Q}
$$

$$
\frac{\partial L}{\partial K}
$$

$$
\frac{\partial L}{\partial V}
$$

进一步通过 Q/K/V projection，gradient 还会继续传到：

$$
W_Q,\ W_K,\ W_V
$$

这就是之后训练 Self-Attention 的基础。


In [21]:
import math

B = 2
H = 4
T = 8
Dh = 16

q = torch.randn(B, H, T, Dh, requires_grad=True)
k = torch.randn(B, H, T, Dh, requires_grad=True)
v = torch.randn(B, H, T, Dh, requires_grad=True)

scores = q @ k.transpose(-2, -1)
scores = scores / math.sqrt(Dh)

attention = torch.softmax(scores, dim=-1)

output = attention @ v

loss = output.pow(2).mean()
loss.backward()

print("loss:", loss.item())

print()
print("Q     :", q.shape)
print("Q.grad:", q.grad.shape)

print()
print("K     :", k.shape)
print("K.grad:", k.grad.shape)

print()
print("V     :", v.shape)
print("V.grad:", v.grad.shape)


loss: 0.18618524074554443

Q     : torch.Size([2, 4, 8, 16])
Q.grad: torch.Size([2, 4, 8, 16])

K     : torch.Size([2, 4, 8, 16])
K.grad: torch.Size([2, 4, 8, 16])

V     : torch.Size([2, 4, 8, 16])
V.grad: torch.Size([2, 4, 8, 16])


## 31. 哪些 Tensor 可以求梯度？

梯度本质上对应连续数值变化。

因此 Autograd 主要用于 floating-point 和 complex Tensor。

例如：

`torch.float32`

`torch.float64`

`torch.bfloat16`

等。

整数 Tensor：

`torch.int64`

通常不能设置：

`requires_grad=True`

这也符合数学直觉。

例如 token ID：

$$
[12,57,301,8]
$$

本质上是离散整数编号。

我们不会直接对：

> token ID 本身

求梯度。

真正被训练的是：

> token 对应的 Embedding Parameters。

这一点等我们学习 Embedding 时会再次出现。


## 32. `.item()` 用来做什么？

如果：

`loss`

是一个 scalar Tensor：

$$
loss.shape=()
$$

我们经常写：

`loss.item()`

得到普通 Python number。

例如：

`print(loss.item())`

这非常适合：

- Logging
- 打印 Loss
- 保存普通数值指标

但是：

`.item()`

得到的已经不是带 Autograd Graph 的 Tensor。

所以不要把 `.item()` 得到的 Python 数值继续拿去构建需要 gradient 的计算。

可以简单记：

`Tensor`

→ 可以处于 Autograd Graph 中

`.item()`

→ 取出普通 Python scalar


## 33. 常见错误

### 错误 1：忘记 `requires_grad=True`

如果 Parameter 不需要 gradient，Autograd 就不会按照预期为它计算 `.grad`。

---

### 错误 2：以为 `backward()` 返回 Gradient

通常不是：

`grad = loss.backward()`

真正的 gradient 通常保存在：

`parameter.grad`

中。

---

### 错误 3：忘记 Gradient Accumulation

多次：

`backward()`

会累积 gradient。

训练循环必须主动清理旧 gradient。

---

### 错误 4：随便使用 `retain_graph=True`

Graph 在 backward 后通常会释放部分中间状态。

不要遇到报错就无脑加：

`retain_graph=True`

应该先理解为什么需要重复 backward。

---

### 错误 5：认为所有 Tensor 的 `.grad` 都会自动保存

默认重点保存 Leaf Tensor 的 gradient。

Intermediate Tensor 如果需要查看 gradient，可以使用：

`retain_grad()`。

---

### 错误 6：Inference 时仍构建完整 Graph

只做推理时通常不需要 gradient tracking。

可以使用：

`torch.no_grad()`

后面还会学习更完整的 inference workflow。

---

### 错误 7：参数更新也进入 Autograd Graph

手写参数更新时通常应该放在：

`with torch.no_grad():`

中。

---

### 错误 8：直接对 Vector 调 `backward()`

最常见、最自然的训练情况是：

scalar loss

↓

`loss.backward()`

对于非 scalar Tensor，需要明确上游 gradient。

---

### 错误 9：把 Gradient Shape 和 Forward Output Shape 混淆

Parameter：

$$
W.shape=(D,H)
$$

通常有：

$$
W.grad.shape=(D,H)
$$

Gradient shape 对应的是 Parameter，而不是整个模型输出。


## 本节总结

### Rule 1：Training 需要 Gradient

Gradient Descent：

$$
\theta
\leftarrow
\theta
-
\eta\nabla_\theta L
$$

---

### Rule 2：`requires_grad=True`

告诉 PyTorch：

> 需要追踪与这个 Tensor 有关的梯度计算。

---

### Rule 3：Forward 构建 Computational Graph

Forward：

$$
x
\rightarrow
operations
\rightarrow
loss
$$

PyTorch 同时记录运算关系。

---

### Rule 4：Backward 使用 Chain Rule

Backward：

$$
loss
\rightarrow
parameters
$$

逐层传播梯度。

---

### Rule 5：Gradient 保存在 `.grad`

例如：

`weight.grad`

表示：

$$
\frac{\partial L}{\partial W}
$$

---

### Rule 6：Parameter 与 Gradient Shape 对应

$$
W.shape
=
W.grad.shape
$$

---

### Rule 7：Gradient 默认累积

多次调用：

`backward()`

gradient 会累积。

因此训练循环需要清 gradient。

---

### Rule 8：Leaf Tensor

Trainable Parameters 通常是 Leaf Tensor。

其 gradient 会保存在：

`.grad`

中。

---

### Rule 9：`torch.no_grad()`

适合：

- 参数更新；
- Validation；
- Inference；

等不需要构建 gradient graph 的场景。

---

### Rule 10：`detach()`

用于从某个 Tensor 开始切断 Autograd Graph。

---

### Rule 11：标准训练核心循环

Clear Gradient

↓

Forward

↓

Loss

↓

Backward

↓

Parameter Gradient

↓

Update Parameters

↓

Next Step

---

### Rule 12：Self-Attention 同样可以自动 Backward

Forward：

$$
Q,K,V
\rightarrow
QK^\top
\rightarrow
Softmax
\rightarrow
AV
\rightarrow
Loss
$$

Backward：

$$
Loss
\rightarrow
V
$$

同时：

$$
Loss
\rightarrow
Q,K
$$

最终继续传到：

$$
W_Q,\ W_K,\ W_V
$$


## Autograd Debug Checklist

以后遇到：

> 为什么没有 gradient？

可以按照下面顺序检查。

### 1. Parameter 是否需要梯度？

检查：

`parameter.requires_grad`

---

### 2. Parameter 是否真的参与了 Loss 的计算？

需要存在：

Parameter

↓

Forward

↓

Loss

这条 Computational Graph。

---

### 3. 是否调用了：

`loss.backward()`

---

### 4. Loss 是否是正常的 Tensor？

不要提前：

`loss.item()`

然后试图对 Python number 做 backward。

---

### 5. 是否在错误的位置使用：

`detach()`

---

### 6. 是否在错误的位置使用：

`torch.no_grad()`

---

### 7. 查看的是不是 Leaf Tensor？

Intermediate Tensor 默认不一定保留 `.grad`。

---

### 8. 是否忘记清理旧 Gradient？

如果 gradient 看起来异常偏大，要检查是否发生了 accumulation。

---

### 9. 是否错误地重复使用旧 Graph？

如果出现：

backward through the graph a second time

之类的错误，首先检查：

> 为什么代码需要在同一个 Forward Graph 上 backward 两次？

而不是直接添加：

`retain_graph=True`。
